In [ ]:

from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor
from scripts.metrics import StatsCollector, StatsHandler


from sampo.base import SAMPO
from sampo.scheduler.genetic import GeneticScheduler
from sampo.schemas.graph import WorkGraph



import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict

# Genetic Algrotihm basic SAMPO  vs Genetic Algorithm with LLMs generated init population

Цель исследования:    
    - Проверить сходимость за сколько был достигнут GAP = 0,   
    Гипотеза: Iter_LLM <  Iter_GA    
    - Оптимальность  
    Гипотеза: makespan_LLM < makespan_GA


In [ ]:
def init_experiment(GA_params, model_names, imprortance):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    for model_name in model_names:
        model = LLMHeuristicScheduler(model_name, **GA_params, 
                                      imprortance=imprortance)
        solvers_dict[model_name] = model
    return solvers_dict

def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        for i, run in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 'run' : i, 'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    return df_res

def run_experiment(solvers_dict, wg, contractors, N_runs):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    experiment_logs = defaultdict(list)
    for i in range(N_runs):
        for solver, model in solvers_dict.items():
            model.schedule(wg, contractors)
            experiment_logs[solver].append(sc.items)
            sc.clear()
    return experiment_logs



GA  = {
    'number_of_generation' : 3,
    'size_of_population' : 60,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 


wg , contractors  = WorkGraph.loadf('wgs/100', '100_0'), contractor(N=5)


#wg , contractors  = WorkGraph.loadf('wgs/small_synth', 'wg_6'), contractor(N=5)
solvers_dict = init_experiment(GA, ['claude_haiku_4.5', ], 11)
experiment_logs = run_experiment(solvers_dict, wg, contractors, 3)


In [ ]:
sns.lineplot(experiment_df(experiment_logs),
              x='Time', y='Makespan', hue='Algorithm', 
              units='run', estimator=None, alpha=0.7)

plt.ylim((25, 150))
plt.xlim((0, 75))

# Проверка гипотезы о весах в ГА

In [ ]:
def tail_share(weights, tail_indices, p_init=1/3):
    W = sum(weights)
    W_tail = sum(weights[tail_indices:])
    return p_init * W_tail / W  # доля в общей популяции


print('Deepseek Chat')
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [0.5] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [0.5] * 9, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [2] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [1] * 9, tail_indices=7) / 0.33 )
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [6] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [5] * 9, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [12] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [10] * 9, tail_indices=7) / 0.33)


print('Reasoner')
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [0.5] * 5, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [2] * 5, tail_indices=7) / 0.33 )
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [6] * 5, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [12] * 5, tail_indices=7) )


In [ ]:
from sampo.schemas.graph import WorkGraph

def init_experiment(GA_params, model_names, imprt):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    for model_name in model_names:
        model = LLMHeuristicScheduler(model_name, **GA_params, imprt=imprt)
        solvers_dict[model_name] = model
    return solvers_dict

def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        if name == 'weight_param':
            continue
        for i, run in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 'run' : i, 'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    df_res['weight_param'] = experiment_logs['weight_param'] 
    return df_res

def run_experiments(GA, wg, weights, contractors, N_runs):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    res_dfs = []
    for weight in weights:
        experiment_logs = defaultdict(list)
        solvers_dict = init_experiment(GA, ['deepseek_chat', 'deepseek_reasoner'], imprt=weight)
        for _ in range(N_runs):
            for solver, model in solvers_dict.items():
                if weight >= weights[0] and solver == 'genetic':
                # Skip other experiments
                    continue
                model.schedule(wg, contractors)
                experiment_logs[solver].append(sc.items)
                experiment_logs['weight_param'] = weight
                sc.clear()
            res_dfs.append(experiment_df(experiment_logs))
    
    return res_dfs


GA  = {
    'number_of_generation' : 150,
    'size_of_population' : 50,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 


weights = (8, 9, 10)


wg , contractors  = WorkGraph.loadf('wgs/small_synth', 'wg_9'), contractor(N=5)
dfs = run_experiments(GA, wg, contractors, 5)


In [ ]:
data = pd.concat(dfs)
data.to_csv('weight_experiment2_df.csv', index=False)

#data.to_csv('weight_experiment2_weights_df.csv', index=False)


# Meta

# GA  = {
#     'number_of_generation' : 150,
#     'size_of_population' : 50,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 


#df_experiment.to_csv('weight_experiment_df.csv', index=False)


# Meta

# GA  = {
#     'number_of_generation' : 100,
#     'size_of_population' : 50,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 



In [ ]:
# from sampo.schemas.graph import WorkGraph
# import os
# from sampo_api import contractor

# maxx, minn = -float('inf'), float('inf')
# for f in os.listdir('wgs/small_synth'):
#         wg , contractors = WorkGraph.loadf('wgs/small_synth', f[:-5]), contractor(N=5)
#         N = len(wg.nodes) 
#         print( f, N)
#         maxx = max(maxx, N)
#         minn = min(minn, N)

# print(maxx, minn)

# Experiment with structure in GA
    - `Classic` GA vs `High Weight` in init_popul vs `Only model generated` init_popul

In [ ]:
from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor
from sampo.schemas.graph import WorkGraph

wg , contractors  = WorkGraph.loadf('wgs/100', '100_0'), contractor(N=10)


def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        for i, run in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 
                               'run' : i, 'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    return df_res



def init_experiment(GA_params, 
                    model_name, 
                    structures = (None, 'onlyGeneratedHeurisitcs'), 
                    imprt = 10):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    for structure in structures:
        model = LLMHeuristicScheduler(model_name, 
                                      **GA_params, 
                                      type_init_pop_structure=structure, 
                                      imprortance=imprt)
        if structure is None:
            structure = 'highWeights'
        solvers_dict[model_name + '_' + structure] = model
    return solvers_dict

def run_experiments(GA, wg, contractors, model_name, N_runs):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    res_dfs = []
    experiment_logs = defaultdict(list)
    solvers_dict = init_experiment(GA, model_name)
    for _ in range(N_runs):
        for solver, model in solvers_dict.items():
            model.schedule(wg, contractors)
            experiment_logs[solver].append(sc.items)
            sc.clear()
            res_dfs.append(experiment_df(experiment_logs))
    return res_dfs


GA  = {
    'number_of_generation' : 100,
    'size_of_population' : 50,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 

dfs = run_experiments(GA, wg, contractors, 'deepseek_reasoner', 4)

In [ ]:
pd.concat(dfs).groupby('Algorithm').Makespan.min()

In [ ]:
pd.concat(dfs).to_csv('structure_experiment_DRChat.csv', index=False)

In [ ]:
str(None)

In [ ]:
# from sampo.schemas.graph import WorkGraph
# WorkGraph.loadf('wgs/100', '100_0')

In [ ]:
GA  = {
    'number_of_generation' : 150,
    'size_of_population' : 50,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 
scheduler = LLMHeuristicScheduler('deepseek_reasoner', 
                                  **GA,  
                                 type_init_pop_structure='onlyGeneratedHeurisitcs')
scheduler.schedule(wg, contractors)


# Эксперимент со структурой популяции


In [ ]:

from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor
from scripts.metrics import StatsCollector, StatsHandler


from sampo.base import SAMPO
from sampo.scheduler.genetic import GeneticScheduler

from sampo.schemas.graph import WorkGraph



import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict



In [ ]:
from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor
from sampo.schemas.graph import WorkGraph

wg , contractors  = WorkGraph.loadf('wgs/small_synth', 'wg_32'), contractor(N=5)


def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        for i, run in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 
                               'run' : i, 'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    return df_res



def init_experiment(GA_params, 
                    model_name, 
                    structures = ('switch_Topological', 
                                  'switch_LFTrand', 
                                  'switch_ALL'), 
                    imprt = 10):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    # for structure in structures:
    #     print(f'Init {structure} for GA')
    #     model = LLMHeuristicScheduler(model_name, 
    #                                   **GA_params, 
    #                                   type_init_pop_structure=structure, 
    #                                   imprortance=imprt)
    #     # if structure is None:
    #     #     structure = 'x'
    #     solvers_dict[model_name + '_' + structure] = model
    return solvers_dict


def run_experiments(GA, wg, contractors, model_name, N_runs):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    res_dfs = []
    experiment_logs = defaultdict(list)
    solvers_dict = init_experiment(GA, model_name)
    for _ in range(N_runs):
        for solver, model in solvers_dict.items():
            model.schedule(wg, contractors)
            experiment_logs[solver].append(sc.items)
            sc.clear()
    res_dfs.append(experiment_df(experiment_logs))
    return res_dfs

# 72
GA  = {
    'number_of_generation' : 2,  
    'size_of_population' : 60,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 

dfs = run_experiments(GA, wg, contractors, 'deepseek_chat', 6)

In [ ]:
pd.concat(dfs)

# Параллельный эскперимент 

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from Experiments.structure import run_one_experiment
from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict
import pandas as pd


def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        for i, makespan_series in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : makespan_series, 
                               'Time' : range(1, len(makespan_series) + 1), 
                               'run' : i, 
                               'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    return df_res



def run_experiments_parallel(GA,
                             size,
                             n_contractors, 
                             model_name, 
                             N_runs, 
                             structures = ('switch_Topological',),
                             max_workers = None):

    # size = 30, 50, 100
    #GA, size, n, structures, model_name = args
    args = [(GA, size, n_contractors, structures, model_name) for _ in range(N_runs)]
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(run_one_experiment, args))
    experiment_logs = defaultdict(list)
    for run_logs in results:
        for solver, logs in run_logs.items():
            experiment_logs[solver].extend(logs)
    return [experiment_df(experiment_logs)]


GA  = {
    'number_of_generation' : 150,  
    'size_of_population' : 50,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 

res_j30_dfs = run_experiments_parallel(GA, 
                                   30,
                                   5,
                                  'deepseek_reasoner', 
                                   N_runs=20, 
                                   max_workers=6)

res_j50_dfs = run_experiments_parallel(GA, 
                                   50,
                                   10,
                                  'deepseek_reasoner', 
                                   N_runs=20, 
                                   max_workers=6)

res_j100_dfs = run_experiments_parallel(GA, 
                                   100,
                                   15,
                                  'deepseek_reasoner', 
                                   N_runs=20, 
                                   max_workers=6)



Can not find native module; switching to default
Can not find native module; switching to default
Can not find native module; switching to default
Can not find native module; switching to default
Can not find native module; switching to default
Can not find native module; switching to default
Can not find native module; switching to default
Init switch_Topological for GA
Init switch_Topological for GA
Init switch_Topological for GA
Init switch_Topological for GA
Init switch_Topological for GA
Init switch_Topological for GA
Genetic optimizing took 52.33502388000488 ms
Genetic optimizing took 55.76586723327637 ms
Genetic optimizing took 69.78082656860352 ms
Genetic optimizing took 45.89271545410156 ms
Genetic optimizing took 58.02798271179199 ms
Genetic optimizing took 58.44831466674805 ms


[SAMPO] [INFO] Toolbox initialization & first population took 52.49810218811035 ms
[SAMPO] [INFO] Toolbox initialization & first population took 55.85980415344238 ms
[SAMPO] [INFO] Toolbox initialization & first population took 69.88215446472168 ms
[SAMPO] [INFO] Toolbox initialization & first population took 45.97902297973633 ms
[SAMPO] [INFO] Toolbox initialization & first population took 58.12191963195801 ms
[SAMPO] [INFO] Toolbox initialization & first population took 58.55894088745117 ms
[SAMPO] [INFO] First population evaluation took 2812.9637241363525 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(74.0,) --
[SAMPO] [INFO] First population evaluation took 2913.8100147247314 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(74.0,) --
[SAMPO] [INFO] First population evaluation took 2954.7479152679443 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(73.0,) --
[SAMPO] [INFO] First population evaluation took 3085.5939388275146 ms
[SAMPO] [

Genetic optimizing took 97.47004508972168 ms


[SAMPO] [INFO] Toolbox initialization & first population took 98.81019592285156 ms
[SAMPO] [INFO] First population evaluation took 3862.2360229492188 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(74.0,) --


Genetic optimizing took 97.15795516967773 ms


...

[SAMPO] [INFO] Toolbox initialization & first population took 73.46129417419434 ms
[SAMPO] [INFO] First population evaluation took 3208.6410522460938 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 143, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] Toolbox initialization & first population took 71.60806655883789 ms


Genetic optimizing took 69.83685493469238 ms


[SAMPO] [INFO] -- Generation 144, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] First population evaluation took 3592.4830436706543 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 145, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 146, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 5, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 147, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 6, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 148, population=50, best fitness=(57.0,) 

Genetic optimizing took 62.04986572265625 ms


[SAMPO] [INFO] Toolbox initialization & first population took 64.09811973571777 ms
[SAMPO] [INFO] -- Generation 13, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 15, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] First population evaluation took 2685.7500076293945 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(74.0,) --
[SAMPO] [INFO] -- Generation 14, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 16, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(74.0,) --
[SAMPO] [INFO] -- Generation 15, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 17, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(73.0,) --
[SAMPO] [INFO] -- Generation 16, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 18, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitnes

In [ ]:
# wg , contractors  = WorkGraph.loadf('wgs/100', '100_0'), contractor(N=10)


In [ ]:
# df_ = pd.concat(res_dfs)
# df_.to_csv('structure_test_DeepseekReasoner_200.csv', index = False)

# #  WorkGraph.loadf('wgs/small_synth', 'wg_32'), contractor(N=5) | Res 1, 110 times, structure_test_DeepseekChat

In [1]:
import pandas as pd
df_ = pd.read_csv('structure_test_DeepseekReasoner_200.csv')

In [ ]:
df_.groupby('Algorithm').Makespan.min()

In [5]:
df_['Time'].value_counts()

Time
1      30
65     30
75     30
74     30
73     30
       ..
130    19
129    19
128    19
126    19
150    19
Name: count, Length: 150, dtype: int64

In [ ]:
# from copy import deepcopy
# from collections import defaultdict
# from pathos.multiprocessing import ProcessingPool

# def run_one_experiment(run_id):
#     solvers_dict = init_experiment(GA, model_name)
#     experiment_logs = defaultdict(list)

#     for solver, model in solvers_dict.items():
#         wg_local = deepcopy(wg)
#         contractors_local = deepcopy(contractors)

#         model.schedule(wg_local, contractors_local)
#         experiment_logs[solver].append("some stats here")

#     return dict(experiment_logs)

# if __name__ == "__main__":
#     pool = ProcessingPool(4)
#     results = pool.map(run_one_experiment, range(N_runs))

In [ ]:
# from copy import deepcopy
# from collections import defaultdict
# from pathos.multiprocessing import ProcessingPool

# # def run_one_experiment(run_id):
# #     solvers_dict = init_experiment(GA, model_name)
# #     experiment_logs = defaultdict(list)

# #     for solver, model in solvers_dict.items():
# #         wg_local = deepcopy(wg)
# #         contractors_local = deepcopy(contractors)

# #         model.schedule(wg_local, contractors_local)
# #         experiment_logs[solver].append("some stats here")

# #     return dict(experiment_logs)

# # if __name__ == "__main__":
# #     pool = ProcessingPool(4)
# #     results = pool.map(run_one_experiment, range(N_runs))